# Consumer Complaint Classification — Gradio Deployment

In [1]:
!pip install -q transformers gradio

## 1. Upload your files
Upload `best_transformer.zip` (the folder zipped) and `label_encoder.pkl`.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving best_transformer.zip to best_transformer.zip


In [3]:
import zipfile, os

zip_name = [f for f in uploaded.keys() if f.endswith(".zip")][0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("best_transformer")

if os.path.isdir("best_transformer/best_transformer"):
    import shutil
    for f in os.listdir("best_transformer/best_transformer"):
        shutil.move(f"best_transformer/best_transformer/{f}", f"best_transformer/{f}")
    shutil.rmtree("best_transformer/best_transformer")

print(os.listdir("best_transformer"))

['tokenizer_config.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'training_args.bin']


## 2. Load model, tokenizer, and label encoder

In [4]:
import pickle
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained("best_transformer").to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("best_transformer")

with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

print(list(label_encoder.classes_))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

['credit_cards', 'credit_reporting', 'debt_collection', 'mortgages_and_loans', 'retail_banking']


## 3. Prediction function

In [5]:
def predict_complaint(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=150).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {label_encoder.classes_[i]: float(probs[i]) for i in range(len(probs))}

## 4. Gradio app

In [6]:
import gradio as gr

demo = gr.Interface(
    fn=predict_complaint,
    inputs=gr.Textbox(lines=6, label="Complaint narrative", placeholder="Describe the issue with your bank, credit card, loan, etc."),
    outputs=gr.Label(num_top_classes=5, label="Predicted category"),
    title="Consumer Complaint Classifier",
    description="Fine-tuned transformer — predicts which team a CFPB-style complaint should be routed to.",
    examples=[
        "I have been trying to dispute an incorrect balance on my credit card statement for months and no one has responded.",
        "My mortgage servicer keeps reporting late payments even though I paid on time every month.",
        "Someone opened a checking account in my name without my permission.",
    ],
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb32ce9c9919ad292e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
